# Employee Attendance & Payroll

In [1]:
import psycopg2
from psycopg2.extras import RealDictCursor
from datetime import datetime
import pandas as pd

## Database connection settings

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "dbname": "employee_attendance_and_payroll",      
    "user": "postgres",
    "password": "--",   
}

In [7]:
class Employee:
    OVERTIME_MULTIPLIER = 1.5  # pay rate multiplier for overtime hours

    def __init__(self, db_config):
        self.conn = psycopg2.connect(**db_config)
        self.cur = self.conn.cursor(cursor_factory=RealDictCursor)
        self.employee_id = None
        self.employee_name = None

    def login(self, user_id, user_password):
        try:
            self.cur.execute(
                "SELECT e_id, e_name FROM employee WHERE e_id=%s AND e_password=%s",
                (user_id, user_password)
            )
            row = self.cur.fetchone()
        except Exception as e:
            print(f"Database error during login: {e}")
            return False

        if row is None:
            return False

        self.employee_id = row["e_id"]
        self.employee_name = row["e_name"]
        print("=" * 60)
        print(f"Employee {row['e_name']} (ID: {row['e_id']}) logged in")
        print("=" * 60)
        return True

    def attendance(self):
        """Toggle clock-in / clock-out. Automatically triggers salary
        calculation for that session when clocking out."""
        self.cur.execute("SELECT e_attendance FROM employee WHERE e_id=%s", (self.employee_id,))
        row = self.cur.fetchone()

        if row["e_attendance"] == "Absent":
            start_time = datetime.now()
            try:
                self.cur.execute(
                    "UPDATE employee SET e_attendance='Present' WHERE e_id=%s", (self.employee_id,))
                self.cur.execute(
                    "INSERT INTO attendance (e_id, attendance, start_time) VALUES (%s, %s, %s)",(self.employee_id, "Present", start_time))
                self.conn.commit()
                print(f"Employee {self.employee_name} clocked IN at {start_time.strftime('%H:%M:%S')}")
            except Exception as e:
                self.conn.rollback()
                print(f"Attendance was not marked, try again: {e}")

        else:  # currently Present -> clock out
            end_time = datetime.now()
            try:
                self.cur.execute(
                    "UPDATE employee SET e_attendance='Absent' WHERE e_id=%s", (self.employee_id,))
                # Close the most recent still-open session for this employee
                self.cur.execute(
                    """UPDATE attendance SET end_time=%s
                       WHERE a_id = (
                           SELECT a_id FROM attendance
                           WHERE e_id=%s AND end_time IS NULL
                           ORDER BY start_time DESC LIMIT 1
                       )
                       RETURNING start_time""", (end_time, self.employee_id))
                closed_row = self.cur.fetchone()
                self.conn.commit()

                if closed_row is None:
                    print("No open attendance session found to close.")
                    return

                print(f"Employee {self.employee_name} clocked OUT at {end_time.strftime('%H:%M:%S')}")
                self._calculate_salary(closed_row["start_time"], end_time)
            except Exception as e:
                self.conn.rollback()
                print(f"Attendance was not marked, try again: {e}")

    def _calculate_salary(self, start_time, end_time):
        """Compute hours worked / overtime / pay for one clock-in-to-out session
        and store it as a row in `salary`."""
        worked_hours = (end_time - start_time).total_seconds() / 3600
        office_hours = min(worked_hours, 8)
        overtime_hours = max(0.0, worked_hours - 8)

        self.cur.execute("SELECT e_salary_ph FROM employee WHERE e_id=%s", (self.employee_id,))
        e_salary_ph = float(self.cur.fetchone()["e_salary_ph"])

        office_ph_salary = office_hours * e_salary_ph
        overtime_ph_salary = overtime_hours * e_salary_ph * self.OVERTIME_MULTIPLIER
        total_salary = office_ph_salary + overtime_ph_salary

        self.cur.execute(
            """INSERT INTO salary
               (e_id, e_salary_ph, office_hours, overtime_hours,
                office_ph_salary, overtime_ph_salary, total_salary)
               VALUES (%s, %s, %s, %s, %s, %s, %s)""",
            (self.employee_id, e_salary_ph, office_hours, overtime_hours,
             office_ph_salary, overtime_ph_salary, total_salary)
        )
        self.conn.commit()

        print(
            f"Session pay calculated -> office hours: {office_hours:.2f}, "
            f"overtime hours: {overtime_hours:.2f}, total: ${total_salary:.2f}"
        )

    def overtime(self):
        self.cur.execute(
            "SELECT COALESCE(SUM(overtime_hours),0) AS total_ot FROM salary WHERE e_id=%s",
            (self.employee_id,)
        )
        total_ot = self.cur.fetchone()["total_ot"]
        print(f"Employee {self.employee_name} has logged {total_ot} overtime hours in total")

    def monthly_report(self):
        """Return (and print) this employee's salary records for the current month."""
        query = """
            SELECT record_date, office_hours, overtime_hours,
                   office_ph_salary, overtime_ph_salary, total_salary
            FROM salary
            WHERE e_id=%s
              AND date_trunc('month', record_date) = date_trunc('month', CURRENT_DATE)
            ORDER BY record_date
        """
        df = pd.read_sql(query, self.conn, params=(self.employee_id,))
        if df.empty:
            print("No salary records for this month yet.")
        else:
            print(df.to_string(index=False))
        return df

    def export_data(self, filename=None):
        df = self.monthly_report()
        if df.empty:
            print("Nothing to export.")
            return
        filename = filename or f"employee_{self.employee_id}_monthly_report.csv"
        df.to_csv(filename, index=False)
        print(f"Report exported to {filename}")

    def close(self):
        self.cur.close()
        self.conn.close()

In [10]:
obj1 = Employee(DB_CONFIG)
user_id = input("Enter your id : ")
user_password = input("Enter your password : ")

try:
    if obj1.login(user_id, user_password):
        while True:
            user_choice = input(
                "What do you want to do : \n"
                " 1. Attendance (clock in/out) \n"
                " 2. Check Overtime \n"
                " 3. See Monthly Report \n"
                " 4. Export Report to CSV \n"
                " 5. Exit \n> "
            )
            if user_choice == "1":
                obj1.attendance()
            elif user_choice == "2":
                obj1.overtime()
            elif user_choice == "3":
                obj1.monthly_report()
            elif user_choice == "4":
                obj1.export_data()
            else:
                print("-" * 60, "\nExiting System ....\n", "-" * 60)
                break
    else:
        print("Employee credentials are not correct ...")
finally:
    obj1.close()

Employee user 2 (ID: 2) logged in
Employee user 2 clocked OUT at 03:02:06
Session pay calculated -> office hours: 0.03, overtime hours: 0.00, total: $0.39
record_date  office_hours  overtime_hours  office_ph_salary  overtime_ph_salary  total_salary
 2026-07-31          0.03             0.0              0.39                 0.0          0.39


C:\Users\User\AppData\Local\Temp\ipykernel_2188\3413582973.py:124: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn, params=(self.employee_id,))


record_date  office_hours  overtime_hours  office_ph_salary  overtime_ph_salary  total_salary
 2026-07-31          0.03             0.0              0.39                 0.0          0.39
Report exported to employee_2_monthly_report.csv


C:\Users\User\AppData\Local\Temp\ipykernel_2188\3413582973.py:124: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn, params=(self.employee_id,))


------------------------------------------------------------ 
Exiting System ....
 ------------------------------------------------------------
